# Hugging Face 모델과 Adapter를 GGUF로 변환하기

**GGUF**는 모델 구조·가중치·tokenizer 정보를 llama.cpp 계열 runtime이 읽기 쉽게 담는 파일 형식이다. Hugging Face 저장소라고 모두 같은 변환 과정을 거치지는 않으므로 먼저 입력 저장소의 종류를 구분한다.

| 입력 저장소 | 들어 있는 것 | 필요한 처리 | 이 노트북 |
|---|---|---|---|
| Full Hugging Face model | config, tokenizer, 전체 Safetensors weight | GGUF 변환 후 필요하면 양자화 | 선택 개요 |
| Adapter-only 저장소 | LoRA 변화량과 adapter 설정 | 정확한 base와 병합한 뒤 GGUF 변환·양자화 | **필수 실습** |
| 이미 GGUF인 저장소 | 변환 또는 양자화가 끝난 GGUF | 변환 없이 Ollama로 직접 실행하거나 등록 | 선택 복습 |

변환, 양자화와 Ollama 등록은 목적이 다른 단계이다.

- **형식 변환**은 Hugging Face 모델을 GGUF 컨테이너로 옮기는 과정이다.
- **양자화**는 weight의 bit 수를 낮춰 파일 크기와 실행 메모리를 줄이는 과정이다.
- **Ollama 등록**은 GGUF와 system prompt 같은 실행 설정을 하나의 model tag로 만드는 과정이다.

이번 필수 경로는 `{HGGING_FACE_ID}/news2stock-qlora` adapter를 정확한 base model과 병합해 `news2stock` GGUF로 만드는 것이다. 앞에서 일반 full model 변환 방법을 짧게 비교한 뒤, adapter 병합부터 Ollama 실행까지 직접 수행한다.

### 변환 과정

`NCSOFT/Llama-VARCO-8B-Instruct` + `{HGGING_FACE_ID}/news2stock-qlora` → `merge_and_unload()` → Hugging Face 전체 모델 → F16 GGUF → Q5_K_M GGUF → Ollama `news2stock`


## 실행 환경과 저장 공간

이 실습은 모델 다운로드, 병합, GGUF 변환과 Ollama 실행이 이어지므로 **RunPod GPU 환경**에서 진행한다. `/workspace`가 Network Volume 또는 Pod 재시작 후에도 유지되는 저장 공간인지 먼저 확인한다.

- GPU: BF16 8B 모델을 올릴 수 있는 A40 48GB 또는 A100 80GB를 권장한다.
- 디스크: base cache, 병합 모델, F16 GGUF, Q5 GGUF와 llama.cpp build를 함께 보관하므로 **최소 60GB, 권장 70GB의 여유 공간**이 필요하다.
- 인증: 공개 저장소만 사용하면 token 없이도 받을 수 있지만 다운로드 제한이나 저장소 정책에 대비해 RunPod에 `HF_TOKEN`을 등록할 수 있다.
- Ollama: `01_ollama.ipynb`의 설치와 server 실행을 먼저 완료한다.

이번 노트북은 대용량 파일을 여러 개 생성한다. 비용을 줄이려면 최종 `news2stock-q5_k_m.gguf`가 정상 동작한 뒤 불필요한 F16 중간 파일과 Hugging Face cache를 별도로 정리한다.


## 선택 개요 1: GGUF My Repo Space로 Full Model 변환

[GGUF My Repo Space](https://huggingface.co/spaces/ggml-org/gguf-my-repo)는 **전체 Hugging Face 모델 저장소**를 입력받아 GGUF 변환과 양자화를 수행하고 결과를 자신의 Hub 저장소에 만드는 UI이다. adapter-only 저장소는 base weight가 없으므로 이번 News2Stock 필수 경로에는 이 Space를 바로 적용하지 않는다.

1. Hugging Face에 로그인한다.
2. `Hub Model ID`에서 변환할 full model 저장소를 선택한다.
3. `Quantization Method`에서 필요한 Q4 또는 Q5 계열을 선택한다.
4. 공개 여부와 대용량 파일 분할 여부를 확인하고 `Submit`을 누른다.
5. 완료된 저장소에서 GGUF 파일명·크기·base model·license를 확인한다.

<img src="https://d.pr/i/rnCiMF+" width="500" />

첫 번째 화면은 `Hub Model ID`와 `Quantization Method`를 지정하는 위치를 보여 준다.

<img src="https://d.pr/i/BzNQqg+" width="500" />

두 번째 화면은 변환 완료 후 새 GGUF 저장소 링크가 나타나는 예시이다. Space UI의 필드와 기본 quantization은 변경될 수 있으므로 실제 실행 시 현재 화면을 기준으로 선택한다.


## 선택 개요 2: llama.cpp로 Full Model 변환

Space 대신 과정을 명령으로 재현하려면 [llama.cpp의 `convert_hf_to_gguf.py`](https://github.com/ggml-org/llama.cpp/blob/master/convert_hf_to_gguf.py)를 사용한다. 입력 디렉터리에는 `config.json`, tokenizer 파일과 전체 Safetensors weight가 함께 있어야 한다.

다음은 일반적인 full Hugging Face model의 명령 구조이다.

```bash
git clone https://github.com/ggml-org/llama.cpp /workspace/llama.cpp
python3 -m pip install -r /workspace/llama.cpp/requirements.txt

python3 /workspace/llama.cpp/convert_hf_to_gguf.py \
  /workspace/models/hf-model \
  --outfile /workspace/models/my-model-f16.gguf \
  --outtype f16
```

`--outtype f16`은 양자화 전 고정밀 GGUF를 만든다. `Q4_K_M`이나 `Q5_K_M`은 이 인자에 넣지 않고, F16 GGUF를 만든 뒤 `llama-quantize`로 별도 적용한다. 뒤의 News2Stock 필수 실습에서는 같은 도구에 실제 병합 모델 경로를 전달해 이 흐름을 실행한다.


## Adapter 저장소와 정확한 Base Model

LoRA adapter는 원래 가중치 전체가 아니라 학습된 변화량만 저장한다. `adapter_config.json`의 `base_model_name_or_path`는 이 변화량을 어느 원본 모델에 더해야 하는지 기록한다.

- [news2stock QLoRA adapter 설정](https://huggingface.co/{HGGING_FACE_ID}/news2stock-qlora/blob/main/adapter_config.json): `NCSOFT/Llama-VARCO-8B-Instruct`를 base로 지정한다.
- [NCSOFT Llama-VARCO-8B-Instruct](https://huggingface.co/NCSOFT/Llama-VARCO-8B-Instruct): Meta Llama 3.1 8B 계열의 전체 모델이다.

다음 코드에서는 저장소 이름을 하드코딩해 추측하지 않고 `PeftConfig`가 실제 adapter 설정을 읽게 한다. 이후 base model 로드에도 이 값을 그대로 사용하므로 학습 때의 모델과 병합 때의 모델이 일치한다.


### 병합 라이브러리 설치

`transformers`는 base model과 tokenizer를 불러오고, `peft`는 LoRA adapter를 결합하고 병합한다. 앞의 QLoRA 실습과 같은 버전을 사용해 저장된 adapter 계약을 유지한다. 설치가 끝난 뒤 kernel 재시작 안내가 나오면 재시작하고 다음 셀부터 실행한다.


In [ ]:
%pip install -U "transformers==4.56.2" "peft==0.17.1" "accelerate>=1,<2" safetensors hf_transfer


### 저장소 ID와 출력 경로 준비

adapter 저장소 ID를 입력으로 두고, 병합된 Hugging Face 모델과 두 단계의 GGUF 파일을 모두 `/workspace/models` 아래에 저장한다. `HF_TOKEN`은 환경변수에서 읽으며 코드나 출력에 값을 직접 남기지 않는다.


In [ ]:
import os
from pathlib import Path

ADAPTER_MODEL_ID = f"{HGGING_FACE_ID}/news2stock-qlora"
MODEL_ROOT = Path("/workspace/models")
MERGED_MODEL_DIR = MODEL_ROOT / "news2stock-merged"
F16_GGUF_PATH = MODEL_ROOT / "news2stock-f16.gguf"
Q5_GGUF_PATH = MODEL_ROOT / "news2stock-q5_k_m.gguf"
HF_TOKEN = os.getenv("HF_TOKEN")

HGGING_FACE_ID = '허깅페이스_ID'


## BF16 Base Model과 Adapter 결합

QLoRA 학습 때는 base model을 4비트로 줄여 VRAM을 절약했지만, **병합할 때는 base model을 4비트로 로드하지 않는다**. 4비트로 이미 반올림된 가중치에 adapter를 합치면 병합 결과의 정밀도가 불필요하게 낮아지고, 이후 GGUF 양자화와 겹쳐 품질 저하가 커질 수 있다.

여기서는 base model을 BF16으로 적재하고 adapter를 연결한다. `PeftModel.from_pretrained()`의 결과는 아직 base와 adapter가 분리된 추론 객체이며, 다음 셀의 `merge_and_unload()`가 두 가중치를 하나의 일반 Transformers 모델로 만든다.


In [ ]:
import torch
from peft import PeftConfig, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer


## Adapter 병합과 전체 Hugging Face 모델 저장

`merge_and_unload()`는 LoRA가 표현한 변화량을 base weight에 실제로 더하고 PEFT adapter layer를 제거한다. `safe_merge=True`는 병합 과정에서 비정상적인 adapter 값을 점검한다. 반환값은 adapter 없이도 불러올 수 있는 일반 `AutoModelForCausalLM` 형태이다.

병합 결과와 tokenizer를 같은 디렉터리에 저장해야 `convert_hf_to_gguf.py`가 모델 구조, 가중치와 tokenization 정보를 함께 읽을 수 있다. 이 셀은 약 16GB 규모의 전체 모델을 생성하므로 완료될 때까지 Pod를 중지하지 않는다.


## 왜 Adapter를 바로 변환하지 않고 먼저 병합하는가

llama.cpp의 `convert_lora_to_gguf.py`와 Ollama의 adapter import처럼 adapter를 별도 GGUF로 다루는 경로도 존재한다. 그러나 이 경로는 실행 시에도 정확히 같은 base model이 필요하고, QLoRA 학습에 사용한 양자화 방식과 runtime의 양자화 방식이 다르면 결과가 달라질 수 있다.

이 노트북에서는 다음 이유로 **BF16 병합 → 전체 모델 GGUF 변환**을 기본 경로로 사용한다.

- Ollama에 하나의 GGUF 파일만 등록하면 되어 base와 adapter의 조합 실수를 줄인다.
- 병합을 고정밀에서 한 뒤 원하는 방식으로 한 번 양자화하므로 변환 순서가 명확하다.
- 다른 PC로 옮길 때 최종 GGUF와 Modelfile만 전달하면 된다.

직접 adapter import는 용량을 더 절약해야 하거나 여러 adapter를 같은 base에 교체해 쓰는 운영 시나리오에서 별도로 선택한다. 관련 제약은 [Ollama 모델 가져오기](https://docs.ollama.com/import)와 [PEFT LoRA 병합 API](https://huggingface.co/docs/peft/main/package_reference/lora)에서 확인할 수 있다.


## llama.cpp 준비

[llama.cpp](https://github.com/ggml-org/llama.cpp)는 Hugging Face 모델을 GGUF로 변환하는 `convert_hf_to_gguf.py`와 GGUF를 다시 양자화하는 `llama-quantize`를 제공한다. 저장소를 `/workspace/llama.cpp`에 내려받고 변환 script 의존성과 CPU용 quantizer를 준비한다.

Ollama가 최종 추론에서 GPU를 사용하므로 이 단계의 llama.cpp build에는 CUDA 옵션이 필수가 아니다. 기존 `/workspace/llama.cpp`가 있다면 새로 clone하는 대신 해당 폴더에서 `git pull` 후 설치·build 명령부터 실행한다.


RunPod image에 build 도구가 없다면 **Web Terminal에서 최초 1회** 다음 명령을 실행한다.

```bash
apt update && apt install -y git cmake build-essential
```


In [ ]:
import subprocess
import sys

LLAMA_CPP_DIR = Path("/workspace/llama.cpp")


### VARCO tokenizer 인식 오류 보완

일부 llama.cpp 버전은 VARCO tokenizer의 pre-tokenizer hash를 자동으로 분류하지 못해 `BPE pre-tokenizer was not recognized` 오류를 발생시킨다. VARCO는 Llama 3 계열 BPE 규칙을 사용하므로 다음 코드는 변환기의 인식 표에 해당 hash와 `llama-bpe` 방식을 연결한다.

이 코드는 모델 weight나 tokenizer 파일을 바꾸지 않고 현재 `/workspace/llama.cpp` 변환기에만 호환 규칙을 추가한다. 이후 llama.cpp가 VARCO를 공식 지원하면 이 보완 단계는 생략할 수 있다.


In [ ]:
from pathlib import Path

CONVERTER_BASE_PATH = LLAMA_CPP_DIR / "conversion" / "base.py"
VARCO_TOKENIZER_HASH = (
    "1baddeb572cd9de2a6d36f2ad0c361490bf5447dafca20afbac625e9d37f18a5"
)

source = CONVERTER_BASE_PATH.read_text(encoding="utf-8")
marker = "        if res is None:\n"
varco_mapping = (
    f'        if chkhsh == "{VARCO_TOKENIZER_HASH}":\n'
    '            # VARCO는 Llama 3.1과 같은 BPE pre-tokenizer 규칙을 사용한다.\n'
    '            res = "llama-bpe"\n\n'
)

if VARCO_TOKENIZER_HASH not in source:
    if marker not in source:
        raise RuntimeError(
            "llama.cpp 구조가 변경되어 자동 패치 위치를 찾지 못했다."
        )

    source = source.replace(marker, varco_mapping + marker, 1)
    CONVERTER_BASE_PATH.write_text(source, encoding="utf-8")
    print("VARCO tokenizer 호환 규칙 추가 완료")
else:
    print("VARCO tokenizer 호환 규칙이 이미 존재함")

F16_GGUF_PATH.unlink(missing_ok=True)
print("재실행 준비 완료:", F16_GGUF_PATH)


## 병합 모델을 F16 GGUF로 변환

`convert_hf_to_gguf.py`는 병합 모델 디렉터리의 config, tokenizer와 Safetensors를 읽어 하나의 GGUF 파일로 묶는다. `--outtype f16`은 먼저 고정밀 중간 파일을 만드는 설정이다.

`Q5_K_M`은 이 변환 script의 `--outtype`에 넣는 값이 아니다. 먼저 F16 GGUF를 만든 뒤 다음 단계에서 `llama-quantize`로 5비트 양자화를 적용한다. 공식 인자와 지원 architecture는 [convert_hf_to_gguf.py](https://github.com/ggml-org/llama.cpp/blob/master/convert_hf_to_gguf.py)에서 확인할 수 있다.


In [ ]:
CONVERT_SCRIPT = LLAMA_CPP_DIR / "convert_hf_to_gguf.py"


## F16 GGUF를 Q5_K_M으로 양자화

양자화는 GGUF 형식 변환과 별개의 단계이다. `Q5_K_M`은 주요 weight를 약 5비트로 표현해 F16보다 파일 크기와 실행 메모리를 줄이면서 Q4 계열보다 품질을 조금 더 보존하는 선택이다.

입력은 방금 만든 F16 GGUF이고 출력은 Ollama에 등록할 최종 `news2stock-q5_k_m.gguf`이다. 양자화 결과는 원본 F16과 별도 파일로 남기므로 문제가 있으면 다른 quantization으로 다시 만들 수 있다. 자세한 형식은 [llama.cpp quantize 문서](https://github.com/ggml-org/llama.cpp/blob/master/tools/quantize/README.md)에서 확인할 수 있다.


In [ ]:
QUANTIZE_BIN = LLAMA_CPP_DIR / "build" / "bin" / "llama-quantize"

quantize_input = str(F16_GGUF_PATH)
quantize_output = str(Q5_GGUF_PATH)
quantize_type = "Q5_K_M"


## Modelfile 작성과 Ollama 등록

`Modelfile`은 Ollama가 사용할 GGUF 경로와 기본 동작을 선언한다. `FROM`에는 이미 양자화한 Q5_K_M 파일을 지정하므로 `ollama create --quantize`를 다시 사용하지 않는다.

GGUF의 현재 정밀도에 따라 등록 명령을 구분한다.

| 입력 GGUF | Ollama 등록 | 이유 |
|---|---|---|
| F16/FP32 | `ollama create --quantize q4_K_M my-model -f Modelfile` | 등록하면서 한 번 양자화한다. |
| 이미 Q4/Q5 | `ollama create my-model -f Modelfile` | 재양자화하면 품질이 더 손상될 수 있다. |

```text
# F16/FP32 GGUF의 Modelfile
FROM /workspace/models/my-model-f16.gguf
```

```bash
ollama create --quantize q4_K_M my-model-q4 -f ./Modelfile
```

```text
# 이미 양자화된 GGUF의 Modelfile
FROM /workspace/models/my-model-q5_k_m.gguf
```

```bash
ollama create my-model-q5 -f ./Modelfile
```

이번 News2Stock 파일은 앞에서 `llama-quantize`로 Q5_K_M을 이미 적용했으므로 두 번째 경로를 사용한다. 자세한 구분은 [Ollama 모델 가져오기](https://docs.ollama.com/import)에서 확인할 수 있다.

이 셀은 `/workspace/models/Modelfile.news2stock`을 만들고 `news2stock`이라는 로컬 모델 tag를 등록한다. 학습 시 사용한 system prompt도 `SYSTEM`에 함께 고정해야 뉴스만 입력해도 출력 계약을 안정적으로 재현할 수 있다. 등록 뒤 `ollama run news2stock`으로 뉴스 원문을 전달하면 파인튜닝한 분석 형식이 유지되는지 확인할 수 있다.


In [ ]:
MODELFILE_PATH = MODEL_ROOT / "Modelfile.news2stock"

SYSTEM_PROMPT = """금융·경제 뉴스를 요약하고 주식 관련 영향을 JSON 객체로만 답한다.
key는 stock_related, summary, positive_stocks, positive_keywords, positive_reasons, negative_stocks, negative_keywords, negative_reasons를 사용한다.
stock_related는 불리언, stocks와 keywords는 리스트, summary와 reasons는 문자열로 작성한다. 값이 없으면 문자열은 빈 문자열, 리스트는 빈 리스트로 작성한다."""


### `news2stock` 모델 생성과 실행

`ollama create`는 Modelfile을 읽어 로컬 모델 tag를 만들고, `ollama run`은 실행 중인 Ollama server에서 그 모델을 추론한다. 이 단계는 앞의 `01_ollama.ipynb`에서 Ollama server가 시작된 상태여야 한다.

생성된 답변에서는 단순 자연어 문장만 보는 것이 아니라 파인튜닝 데이터에서 사용한 뉴스 분석 구조와 한국어 품질이 유지되는지 확인한다. 외부 PC에서는 앞서 `03_ollama_deployment.ipynb`에서 배운 API 호출 코드를 재사용한다.


## 변환 결과 확인과 외부 API 코드 재사용

다음 항목을 확인하면 병합부터 Ollama 등록까지의 흐름이 완료된다.

- `ollama list`에 `news2stock`이 표시되는가?
- `ollama show --modelfile news2stock`의 `FROM`, `SYSTEM`과 chat template가 의도한 설정인가?
- 학습에 사용하지 않은 새 뉴스에서도 요구한 분석 구조가 유지되는가?
- 같은 prompt에서 원래 Transformers 추론과 양자화된 Ollama 추론의 핵심 판단이 유지되는가?
- 한글·특수문자와 긴 문맥에서 tokenizer와 출력 구조가 깨지지 않는가?
- 파일 크기, 실행 메모리와 응답 속도가 목표 장비 범위에 들어오는가?
- base model과 학습 데이터의 license가 GGUF 재배포를 허용하는가?

양자화 모델은 크기와 메모리를 줄이는 대신 답변이 조금 달라질 수 있다. 한 문장만 보고 완료로 판단하지 않고 여러 뉴스에서 구조 준수, 내용 정확성, 속도와 메모리를 함께 비교한다. RunPod 외부에서는 앞서 `03_ollama_deployment.ipynb`에서 배운 API 호출 코드를 그대로 사용하고 `OLLAMA_MODEL`만 `news2stock`으로 바꾼다. port와 접근 범위는 [RunPod Ollama 가이드](https://docs.runpod.io/tutorials/pods/run-ollama)의 `11434` 설정을 유지한다.


## Q5_K_M GGUF를 Hugging Face Hub에 업로드

Ollama의 `news2stock` tag는 현재 Pod 안에 등록된 실행 이름이고 Hugging Face에 직접 업로드하는 대상은 아니다. Hub에는 다른 환경에서도 다시 내려받을 수 있도록 최종 `news2stock.Q5_K_M.gguf`와 실행 설정을 담은 `Modelfile`을 올린다.

`HfApi.create_repo()`는 모델 저장소가 없을 때 생성하고, `upload_file()`은 로컬 파일을 해당 저장소에 전송한다. `HF_TOKEN`에는 write 권한이 필요하며 GGUF 파일이 크므로 업로드가 끝날 때까지 Pod와 kernel을 유지한다. Ollama 등록과 Hub 업로드는 독립된 작업이므로 Q5_K_M 파일만 완성됐다면 어느 쪽을 먼저 실행해도 된다.


In [ ]:
import os

from huggingface_hub import HfApi

REPO_ID = f"{HGGING_FACE_ID}/news2stock-gguf"
